# 30 — V4 validation diagnostics

Runs the frozen V4 checkpoint on validation only and records frame-position, year, category, sequence-pattern, calibration, and hardest-error diagnostics. The official test is not loaded or evaluated.


In [1]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path

PACKAGE_VERSION = "0.1.10"
VALIDATION_FRACTION = 0.20
VALIDATION_SEED = 20260913
YEARS = tuple(range(2013, 2023))
VARIABLES = ("DBZ", "KDP", "RHOHV", "VEL", "WIDTH", "ZDR")
CHANNEL_ORDER = tuple(
    f"{variable}_sweep_{sweep}"
    for variable in VARIABLES
    for sweep in range(2)
)
FILE_BATCH_SIZE = 8
NUM_WORKERS = 12
SELECTED_THRESHOLD = 0.9532915353775024
EXPECTED_CHECKPOINT_EPOCH = 23

BACKUP_ROOT = Path("/content/drive/MyDrive/TorNet_Backup")
PACKAGE_PATH = BACKUP_ROOT / "packages" / f"tornet_detection-{PACKAGE_VERSION}-py3-none-any.whl"
MANIFESTS_ROOT = BACKUP_ROOT / "manifests"
NORMALIZATION_PATH = BACKUP_ROOT / "experiments" / "all_year_all6_baseline_v1" / "normalization.json"
EXPERIMENT_DIRECTORY = BACKUP_ROOT / "experiments" / "spatiotemporal_bidirectional_all_year_v4"
BEST_CHECKPOINT_PATH = EXPERIMENT_DIRECTORY / "best_model.pt"
VALIDATION_DIAGNOSTICS_PATH = (
    EXPERIMENT_DIRECTORY
    / "validation_diagnostics.json"
)
VALIDATION_PREDICTIONS_PATH = (
    EXPERIMENT_DIRECTORY
    / "validation_predictions.parquet"
)
EXTRACTION_ROOT = Path("/content/tornet_bidirectional_v4_validation_diagnostics")

for path in (
    PACKAGE_PATH,
    MANIFESTS_ROOT,
    NORMALIZATION_PATH,
    BEST_CHECKPOINT_PATH,
):
    if not path.exists():
        raise FileNotFoundError(path)

if (
    VALIDATION_DIAGNOSTICS_PATH.exists()
    or VALIDATION_PREDICTIONS_PATH.exists()
):
    raise FileExistsError(
        "Validation diagnostics already exist"
    )

print("checkpoint:", BEST_CHECKPOINT_PATH)
print("frozen threshold:", SELECTED_THRESHOLD)


checkpoint: /content/drive/MyDrive/TorNet_Backup/experiments/spatiotemporal_bidirectional_all_year_v4/best_model.pt
frozen threshold: 0.9532915353775024


In [3]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "netCDF4>=1.7",
        "scikit-learn>=1.5",
    ],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--force-reinstall",
        str(PACKAGE_PATH),
    ],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--no-deps', '--force-reinstall', '/content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.10-py3-none-any.whl'], returncode=0)

In [4]:
import datetime
import json
import random
import shutil
import tarfile
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import (
    DataLoader,
    Dataset,
)

import tornado_detection
from tornado_detection.data import (
    assign_model_splits,
    load_canonical_frame_index,
    read_spatiotemporal_netcdf_file,
)
from tornado_detection.models import (
    SpatiotemporalTornadoDetector,
)

if tornado_detection.__version__ != PACKAGE_VERSION:
    raise RuntimeError(
        "Unexpected package version: "
        f"{tornado_detection.__version__}"
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "Select an A100 GPU runtime and run all cells"
    )

if not torch.cuda.is_bf16_supported():
    raise RuntimeError(
        "The selected GPU does not support bfloat16"
    )


device = torch.device("cuda")

normalization = json.loads(
    NORMALIZATION_PATH.read_text()
)

expected_normalization = {
    "package_version": "0.1.7",
    "years": list(YEARS),
    "model_split": "train",
    "validation_fraction": (
        VALIDATION_FRACTION
    ),
    "validation_seed": VALIDATION_SEED,
    "training_frame_count": 547_672,
    "training_file_count": 136_918,
    "training_positive_frame_count": 18_230,
    "variables": list(VARIABLES),
    "channel_order": list(CHANNEL_ORDER),
    "tensor_shape": [
        120,
        240,
        len(CHANNEL_ORDER),
    ],
}

mismatches = {
    key: {
        "expected": expected,
        "actual": normalization.get(key),
    }
    for key, expected
    in expected_normalization.items()
    if normalization.get(key) != expected
}

if mismatches:
    raise AssertionError(
        "Normalization provenance mismatch: "
        f"{mismatches}"
    )

means = np.asarray(
    normalization["means"],
    dtype=np.float32,
).reshape(
    len(VARIABLES),
    2,
).T
stds = np.asarray(
    normalization["standard_deviations"],
    dtype=np.float32,
).reshape(
    len(VARIABLES),
    2,
).T

assert means.shape == (2, 6)
assert stds.shape == (2, 6)
assert np.isfinite(means).all()
assert np.isfinite(stds).all()
assert (stds > 0).all()

canonical = load_canonical_frame_index(
    MANIFESTS_ROOT
)
assigned = assign_model_splits(
    canonical,
    validation_fraction=VALIDATION_FRACTION,
    seed=VALIDATION_SEED,
)

validation_rows = (
    assigned.loc[
        assigned["model_split"].eq(
            "validation"
        )
    ]
    .sort_values(
        [
            "archive_member",
            "frame_index",
        ]
    )
)

assert len(validation_rows) == 138_892
assert int(
    validation_rows["frame_label"].sum()
) == 4_200

if not validation_rows.groupby(
    "archive_member"
).size().eq(4).all():
    raise AssertionError(
        "Every validation file must "
        "contain four frames"
    )

validation_records = [
    (
        member,
        group["frame_label"]
        .astype(np.uint8)
        .to_numpy(),
    )
    for member, group
    in validation_rows.groupby(
        "archive_member",
        sort=True,
    )
]

assert len(validation_records) == 34_723

required_by_year = {
    year: set(
        validation_rows.loc[
            validation_rows["year"].eq(year),
            "archive_member",
        ].unique()
    )
    for year in YEARS
}

print(
    "GPU:",
    torch.cuda.get_device_name(0),
)
print("torch:", torch.__version__)
print(
    "validation files/frames:",
    len(validation_records),
    len(validation_rows),
)
print("official test loaded: no")


GPU: NVIDIA A100-SXM4-40GB
torch: 2.11.0+cu128
validation files/frames: 34723 138892
official test loaded: no


In [5]:
if EXTRACTION_ROOT.exists():
    shutil.rmtree(EXTRACTION_ROOT)

EXTRACTION_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

staging = []

for year in YEARS:
    drive_archive = (
        BACKUP_ROOT
        / f"tornet_{year}.tar.gz"
    )
    local_archive = Path(
        f"/content/tornet_{year}.tar.gz"
    )

    if not drive_archive.is_file():
        raise FileNotFoundError(
            drive_archive
        )

    if local_archive.exists():
        local_archive.unlink()

    copy_started = (
        time.perf_counter()
    )

    shutil.copyfile(
        drive_archive,
        local_archive,
    )

    copy_seconds = (
        time.perf_counter()
        - copy_started
    )

    required = required_by_year[year]
    extracted = set()
    extract_started = (
        time.perf_counter()
    )

    with tarfile.open(
        local_archive,
        mode="r:gz",
    ) as archive:
        for member in archive:
            if (
                not member.isfile()
                or member.name
                not in required
            ):
                continue

            destination = (
                EXTRACTION_ROOT
                / member.name
            )
            destination.parent.mkdir(
                parents=True,
                exist_ok=True,
            )

            source_file = (
                archive.extractfile(
                    member
                )
            )

            if source_file is None:
                raise RuntimeError(
                    member.name
                )

            with (
                source_file,
                destination.open("wb")
                as output_file,
            ):
                shutil.copyfileobj(
                    source_file,
                    output_file,
                    length=1024 * 1024,
                )

            extracted.add(
                member.name
            )

    extraction_seconds = (
        time.perf_counter()
        - extract_started
    )
    missing = required - extracted

    if missing:
        raise RuntimeError(
            f"Year {year} missing: "
            f"{sorted(missing)[:10]}"
        )

    local_archive.unlink()

    staging.append(
        {
            "year": year,
            "file_count": len(
                extracted
            ),
            "copy_seconds": (
                copy_seconds
            ),
            "extraction_seconds": (
                extraction_seconds
            ),
        }
    )

    print(
        f"year={year} "
        f"staged_files="
        f"{len(extracted):,} "
        f"copy={copy_seconds:.1f}s "
        f"extract="
        f"{extraction_seconds:.1f}s"
    )

print(
    "total staged files:",
    sum(
        row["file_count"]
        for row in staging
    ),
)

staged_bytes = sum(
    path.stat().st_size
    for path
    in EXTRACTION_ROOT.rglob("*.nc")
)

print(
    "staged GiB:",
    round(
        staged_bytes / 1024**3,
        3,
    ),
)

year=2013 staged_files=734 copy=39.2s extract=9.6s
year=2014 staged_files=3,474 copy=165.2s extract=45.4s
year=2015 staged_files=3,997 copy=318.8s extract=57.6s
year=2016 staged_files=4,088 copy=180.1s extract=52.4s
year=2017 staged_files=3,501 copy=262.3s extract=45.2s
year=2018 staged_files=2,916 copy=169.1s extract=38.6s
year=2019 staged_files=4,024 copy=172.2s extract=61.0s
year=2020 staged_files=3,507 copy=267.2s extract=54.4s
year=2021 staged_files=3,764 copy=261.3s extract=60.4s
year=2022 staged_files=4,718 copy=251.0s extract=65.4s
total staged files: 34723
staged GiB: 25.903


In [6]:
class SequenceDataset(Dataset):
    def __init__(
        self,
        file_records,
        root,
        channel_means,
        channel_stds,
    ):
        self.records = file_records
        self.root = root
        self.means = channel_means.reshape(
            1,
            2,
            6,
            1,
            1,
        )
        self.stds = channel_stds.reshape(
            1,
            2,
            6,
            1,
            1,
        )

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        (
            member,
            expected_labels,
        ) = self.records[index]

        sequence = (
            read_spatiotemporal_netcdf_file(
                self.root / member,
                variables=VARIABLES,
            )
        )

        np.testing.assert_array_equal(
            sequence.labels,
            expected_labels,
        )

        normalized = (
            (
                sequence.values
                - self.means
            )
            / self.stds
        ).astype(
            np.float32,
            copy=False,
        )

        return (
            torch.from_numpy(normalized),
            torch.from_numpy(
                sequence.finite_mask
            ),
            torch.from_numpy(
                sequence.range_folded_mask
            ),
            torch.from_numpy(
                sequence.coordinates
            ),
            torch.from_numpy(
                sequence.labels.astype(
                    np.float32
                )
            ),
        )



validation_dataset = SequenceDataset(
    validation_records,
    EXTRACTION_ROOT,
    means,
    stds,
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=FILE_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)

model = SpatiotemporalTornadoDetector(
    pooling_topk_fraction=0.05,
    bidirectional_temporal=True,
).to(device)
checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=device,
    weights_only=False,
)

if checkpoint.get("artifact_kind") != "spatiotemporal_bidirectional_all_year_v4_best_model":
    raise AssertionError("Unexpected checkpoint artifact")

if int(checkpoint["epoch"]) != EXPECTED_CHECKPOINT_EPOCH:
    raise AssertionError(
        f"Expected epoch {EXPECTED_CHECKPOINT_EPOCH}; "
        f"found {checkpoint['epoch']}"
    )

model.load_state_dict(checkpoint["model_state_dict"])

train_positive_count = 18_230
train_negative_count = 547_672 - train_positive_count
criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        train_negative_count / train_positive_count,
        device=device,
    )
)

print("checkpoint epoch:", checkpoint["epoch"])
print(
    "parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}",
)


checkpoint epoch: 23
parameters: 2,970,049


In [7]:
def move_batch_to_device(batch):
    (
        values,
        finite_mask,
        range_folded_mask,
        coordinates,
        targets,
    ) = batch

    return (
        values.to(
            device,
            non_blocking=True,
        ),
        finite_mask.to(
            device,
            non_blocking=True,
        ),
        range_folded_mask.to(
            device,
            non_blocking=True,
        ),
        coordinates.to(
            device,
            non_blocking=True,
        ),
        targets.to(
            device,
            non_blocking=True,
        ),
    )


def evaluate(loader):
    model.eval()

    labels = []
    probabilities = []
    loss_sum = 0.0
    count = 0

    with torch.no_grad():
        for batch in loader:
            (
                values,
                finite_mask,
                range_folded_mask,
                coordinates,
                targets,
            ) = move_batch_to_device(
                batch
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
            ):
                logits = model(
                    values,
                    finite_mask,
                    range_folded_mask,
                    coordinates,
                )
                loss = criterion(
                    logits,
                    targets,
                )

            size = int(
                targets.numel()
            )
            loss_sum += (
                float(loss.cpu())
                * size
            )
            count += size

            labels.extend(
                targets.cpu()
                .numpy()
                .reshape(-1)
                .tolist()
            )
            probabilities.extend(
                logits.sigmoid()
                .float()
                .cpu()
                .numpy()
                .reshape(-1)
                .tolist()
            )

    labels = np.asarray(
        labels,
        dtype=np.int64,
    )
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    return (
        loss_sum / count,
        float(
            average_precision_score(
                labels,
                probabilities,
            )
        ),
        float(
            roc_auc_score(
                labels,
                probabilities,
            )
        ),
        labels,
        probabilities,
    )




In [8]:
validation_loss, validation_pr_auc, validation_roc_auc, labels, probabilities = evaluate(
    validation_loader
)

predictions = (
    probabilities >= SELECTED_THRESHOLD
).astype(np.int64)

frame_results = (
    validation_rows.copy()
    .reset_index(drop=True)
)
frame_results["probability"] = probabilities
frame_results["prediction"] = predictions
frame_results["error"] = predictions - labels

patterns = (
    frame_results.groupby(
        "archive_member"
    )["frame_label"]
    .apply(
        lambda values: "".join(
            str(int(value))
            for value in values
        )
    )
)
frame_results[
    "sequence_label_pattern"
] = frame_results[
    "archive_member"
].map(patterns)


def summarize(rows):
    y = rows[
        "frame_label"
    ].to_numpy(dtype=np.int64)
    probability = rows[
        "probability"
    ].to_numpy(dtype=np.float64)
    prediction = rows[
        "prediction"
    ].to_numpy(dtype=np.int64)

    precision = float(
        precision_score(
            y,
            prediction,
            zero_division=0,
        )
    )
    recall = float(
        recall_score(
            y,
            prediction,
            zero_division=0,
        )
    )
    has_both_labels = (
        len(np.unique(y)) == 2
    )

    return {
        "frame_count": int(len(rows)),
        "positive_count": int(y.sum()),
        "prevalence": float(y.mean()),
        "pr_auc": (
            float(
                average_precision_score(
                    y,
                    probability,
                )
            )
            if has_both_labels
            else None
        ),
        "roc_auc": (
            float(
                roc_auc_score(
                    y,
                    probability,
                )
            )
            if has_both_labels
            else None
        ),
        "precision": precision,
        "recall": recall,
        "f1": float(
            2
            * precision
            * recall
            / max(
                precision + recall,
                1e-12,
            )
        ),
        "mean_probability": float(
            probability.mean()
        ),
    }


def grouped(column):
    return {
        str(key): summarize(rows)
        for key, rows
        in frame_results.groupby(
            column,
            sort=True,
        )
    }


frame_results[
    "probability_bin"
] = pd.cut(
    frame_results["probability"],
    bins=np.linspace(0.0, 1.0, 11),
    include_lowest=True,
)

calibration = [
    {
        "bin": str(interval),
        "frame_count": int(len(rows)),
        "mean_probability": float(
            rows["probability"].mean()
        ),
        "observed_prevalence": float(
            rows["frame_label"].mean()
        ),
    }
    for interval, rows
    in frame_results.groupby(
        "probability_bin",
        observed=True,
    )
]

error_columns = [
    "frame_id",
    "archive_member",
    "year",
    "category",
    "event_group_id",
    "frame_index",
    "frame_time_utc",
    "frame_label",
    "probability",
    "prediction",
    "dbz_finite_fraction",
    "vel_finite_fraction",
    "range_folded_fraction",
    "sequence_label_pattern",
]

hard_false_positives = (
    frame_results.loc[
        frame_results["error"].eq(1),
        error_columns,
    ]
    .sort_values(
        "probability",
        ascending=False,
    )
    .head(100)
    .to_dict("records")
)

hard_false_negatives = (
    frame_results.loc[
        frame_results["error"].eq(-1),
        error_columns,
    ]
    .sort_values(
        "probability",
        ascending=True,
    )
    .head(100)
    .to_dict("records")
)

diagnostics = {
    "artifact_kind": (
        "spatiotemporal_bidirectional_"
        "all_year_v4_validation_diagnostics"
    ),
    "created_at_utc": (
        datetime.datetime.now(
            datetime.timezone.utc
        ).isoformat()
    ),
    "package_version": PACKAGE_VERSION,
    "checkpoint_epoch": int(
        checkpoint["epoch"]
    ),
    "selected_threshold": (
        SELECTED_THRESHOLD
    ),
    "official_test_evaluated": False,
    "overall": summarize(
        frame_results
    ),
    "validation_loss": (
        validation_loss
    ),
    "by_year": grouped("year"),
    "by_category": grouped("category"),
    "by_frame_index": grouped(
        "frame_index"
    ),
    "by_sequence_label_pattern": (
        grouped(
            "sequence_label_pattern"
        )
    ),
    "calibration": calibration,
    "hard_false_positives": (
        hard_false_positives
    ),
    "hard_false_negatives": (
        hard_false_negatives
    ),
    "staging": staging,
}

frame_results.drop(
    columns=["probability_bin"]
).to_parquet(
    VALIDATION_PREDICTIONS_PATH,
    index=False,
)

VALIDATION_DIAGNOSTICS_PATH.write_text(
    json.dumps(
        diagnostics,
        indent=2,
        sort_keys=True,
        default=str,
    )
    + "\n"
)

print(
    json.dumps(
        diagnostics,
        indent=2,
        sort_keys=True,
        default=str,
    )
)
print(
    "wrote:",
    VALIDATION_DIAGNOSTICS_PATH,
)
print(
    "wrote:",
    VALIDATION_PREDICTIONS_PATH,
)


{
  "artifact_kind": "spatiotemporal_bidirectional_all_year_v4_validation_diagnostics",
  "by_category": {
    "NUL": {
      "f1": 0.0,
      "frame_count": 86240,
      "mean_probability": 0.022087146530519706,
      "positive_count": 0,
      "pr_auc": null,
      "precision": 0.0,
      "prevalence": 0.0,
      "recall": 0.0,
      "roc_auc": null
    },
    "TOR": {
      "f1": 0.6641801353778146,
      "frame_count": 9032,
      "mean_probability": 0.57178802621028,
      "positive_count": 4200,
      "pr_auc": 0.7985366411006196,
      "precision": 0.7910496873971701,
      "prevalence": 0.4650132860938884,
      "recall": 0.5723809523809524,
      "roc_auc": 0.787815481117944
    },
    "WRN": {
      "f1": 0.0,
      "frame_count": 43620,
      "mean_probability": 0.09994129131916178,
      "positive_count": 0,
      "pr_auc": null,
      "precision": 0.0,
      "prevalence": 0.0,
      "recall": 0.0,
      "roc_auc": null
    }
  },
  "by_frame_index": {
    "0": {
      "f1"

In [9]:
shutil.rmtree(EXTRACTION_ROOT)

assert not EXTRACTION_ROOT.exists()
assert BEST_CHECKPOINT_PATH.is_file()
assert VALIDATION_DIAGNOSTICS_PATH.is_file()
assert VALIDATION_PREDICTIONS_PATH.is_file()

print(
    "Removed all Colab-local "
    "validation data"
)
print(
    "Preserved:",
    VALIDATION_DIAGNOSTICS_PATH,
)
print(
    "Preserved:",
    VALIDATION_PREDICTIONS_PATH,
)


Removed all Colab-local validation data
Preserved: /content/drive/MyDrive/TorNet_Backup/experiments/spatiotemporal_bidirectional_all_year_v4/validation_diagnostics.json
Preserved: /content/drive/MyDrive/TorNet_Backup/experiments/spatiotemporal_bidirectional_all_year_v4/validation_predictions.parquet
